# Task 3: Soft MoE Optuna Search

Hyperparameter optimization for soft Mixture-of-Experts (Plan 6 §3.5).
Tunes fine-tuning learning rate, softmax temperature, classification weight $\lambda_3$, and balance weight $\lambda_4$.

In [ ]:
# 1. Config
N_TRIALS = 15
OPTUNA_EPOCHS = 12
STUDY_NAME = "task3_soft_moe"
STORAGE_DB = "sqlite:///task3_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path: sys.path.insert(0, str(RESEARCH_ROOT))
from constants import CONFIGS_ROOT
from src.device_utils import get_device
from src.datasets import PetDataset
from src.models_classifier import CorruptionClassifier, SoftMoERestorer
from src.models_dae import ConvDAE
from src.train_loop import train_one_epoch_moe, validate_moe, set_seed
from src.checkpoint_utils import load_checkpoint
set_seed(SEED)
device = get_device()
checkpoint_dir = RESEARCH_ROOT / "checkpoints"
gate_path = checkpoint_dir / "task2_classifier_best.pt"
expert_paths = {"salt": checkpoint_dir / "task2b_specialist_salt_best.pt",
                "blur": checkpoint_dir / "task2b_specialist_blur_best.pt",
                "occlusion": checkpoint_dir / "task2b_specialist_occlusion_best.pt"}
required = [gate_path, *expert_paths.values()]
missing = [str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError("Train Tasks 2a and 2b before MoE search: " + ", ".join(missing))


In [ ]:
# 3. Objective Function
train_dataset = PetDataset(mode="train", corruption_mode="all", return_label=True, seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="all", return_label=True, seed=SEED)

def objective(trial: optuna.Trial) -> float:
    finetune_lr = trial.suggest_float("finetune_lr", 1e-5, 1e-4, log=True)
    temperature = trial.suggest_float("temperature", 0.5, 2.0)
    lambda3 = trial.suggest_float("lambda3", 0.01, 0.5)
    lambda4 = trial.suggest_float("lambda4", 1e-3, 1e-1, log=True)
    alpha = trial.suggest_float("alpha", 0.5, 0.95)
    lambdas = {"lambda1": alpha, "lambda2": 1-alpha,
               "lambda3": lambda3, "lambda4": lambda4}
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    gate_state = torch.load(gate_path, map_location=device, weights_only=False)["model_state_dict"]
    gate = CorruptionClassifier(base_channels=gate_state["conv1.0.weight"].shape[0]).to(device)
    gate.load_state_dict(gate_state)
    specialists = []
    for path in expert_paths.values():
        state = torch.load(path, map_location=device, weights_only=False)["model_state_dict"]
        model = ConvDAE(base_channels=state["enc1.0.weight"].shape[0],
                        latent_dim=state["fc_z.weight"].shape[0]).to(device)
        model.load_state_dict(state)
        specialists.append(model)
    moe = SoftMoERestorer(gate, *specialists, temperature=temperature).to(device)
    optimizer = torch.optim.Adam(moe.parameters(), lr=finetune_lr)
    best_loss = float("inf")
    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_moe(moe, train_loader, optimizer, device, lambdas)
        metrics = validate_moe(moe, val_loader, device, lambdas)
        best_loss = min(best_loss, metrics["val_loss"])
        trial.set_user_attr(f"epoch_{epoch}_routing", metrics["avg_expert_weights"].tolist())
        trial.report(metrics["val_loss"], step=epoch)
        if trial.should_prune(): raise optuna.TrialPruned()
    return best_loss


In [ ]:
# 4. Optimization & Export
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

cfg_path = CONFIGS_ROOT / "task3_best_config.yaml"
with open(cfg_path, "w") as f:
    yaml.dump({
        "task": "task3_soft_moe",
        "best_trial_number": int(study.best_trial.number),
        "best_value": float(study.best_value),
        "params": study.best_params,
    }, f, indent=2)

print(f"Best MoE config saved to {cfg_path}")